In [1]:
%pip -q install gradio pyngrok huggingface_hub

In [ ]:
!nvidia-smi

In [ ]:
import os
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# Skip the install if it already succeeded in this session
try:
    import llama_cpp
    print("llama-cpp-python already installed:", llama_cpp.__version__)
except ImportError:
    %pip install --only-binary=:all: --no-cache-dir llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
import os, shutil, sqlite3
from huggingface_hub import list_repo_files, hf_hub_download

REPO_ID = "HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive"

# /kaggle/working persists between sessions if you set:
# Session options -> Persistence -> "Files only"  (limit ~20 GB)
BASE_DIR = "/kaggle/working/LocalLLM"
MODEL_DIR = os.path.join(BASE_DIR, "models", "qwen3.6-35b-a3b")
CHAT_DB = os.path.join(BASE_DIR, "chat_history.sqlite3")
os.makedirs(MODEL_DIR, exist_ok=True)

# ---- Model: reuse if already downloaded, else fetch the XS quant ----
existing = [f for f in os.listdir(MODEL_DIR) if f.endswith(".gguf")]
if existing:
    MODEL_PATH = os.path.join(MODEL_DIR, existing[0])
    print("Model already on disk, skipping download:", MODEL_PATH)
else:
    ggufs = [f for f in list_repo_files(REPO_ID) if f.endswith(".gguf")]
    xs = [f for f in ggufs if "XS" in f.upper()]
    if not xs:
        raise FileNotFoundError(f"No XS quant found. Available: {ggufs}")
    fname = next((f for f in xs if "IQ4_XS" in f.upper()), xs[0])
    print("Downloading", fname, "(~18 GB, first run only)...")
    MODEL_PATH = hf_hub_download(
        repo_id=REPO_ID, filename=fname, local_dir=MODEL_DIR
    )
    shutil.rmtree(os.path.join(MODEL_DIR, ".cache"), ignore_errors=True)  # save space

# ---- Chat DB: create if missing, keep if present ----
with sqlite3.connect(CHAT_DB) as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

print("Model file:", MODEL_PATH)
print("Chat database:", CHAT_DB)
print("Disk used in /kaggle/working: %.1f GB" % (shutil.disk_usage("/kaggle/working").used / 1024**3))

In [ ]:

import sqlite3
import uuid
import gradio as gr

SYSTEM_PROMPT = (
    "You are a helpful AI assistant. "
    "Give clear, accurate and thoughtful answers."
)

# ---------------- Database helpers ----------------

def get_conversations():
    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT conversation_id, MAX(id) AS last_id
            FROM messages
            GROUP BY conversation_id
            ORDER BY last_id DESC
        """).fetchall()
    return [row[0] for row in rows]


def get_messages(cid):
    if not cid:
        return []

    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT role, content
            FROM messages
            WHERE conversation_id = ?
            ORDER BY id
        """, (cid,)).fetchall()

    return [
        {"role": role, "content": content}
        for role, content in rows
        if role in ("user", "assistant")
    ]


def save_message(cid, role, content):
    with sqlite3.connect(CHAT_DB) as con:
        con.execute("""
            INSERT INTO messages (conversation_id, role, content)
            VALUES (?, ?, ?)
        """, (cid, role, content))


# ---------------- Chat actions ----------------

def new_conversation():
    cid = str(uuid.uuid4())[:8]
    choices = get_conversations()

    if cid not in choices:
        choices.insert(0, cid)

    return cid, [], gr.update(choices=choices, value=cid)


def load_conversation(cid):
    if not cid:
        return "", []
    return cid, get_messages(cid)


import time, re, subprocess
from llama_cpp.llama_chat_format import Jinja2ChatFormatter

CONTEXT_RESERVE = 6144  # room kept free for the reply when trimming old history

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def vram_usage():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.used,memory.total",
             "--format=csv,noheader,nounits"], text=True)
        parts = []
        for line in out.strip().splitlines():
            i, used, total = [x.strip() for x in line.split(",")]
            parts.append(f"GPU{i}: {used}/{total} MB")
        return " | ".join(parts)
    except Exception as e:
        return f"n/a ({e})"


def build_tokens(history):
    meta = llm.metadata
    eos = llm.detokenize([llm.token_eos()]).decode(errors="ignore")
    bos_id = llm.token_bos()
    bos = llm.detokenize([bos_id]).decode(errors="ignore") if bos_id >= 0 else ""
    fmt = Jinja2ChatFormatter(
        template=meta["tokenizer.chat_template"],
        eos_token=eos, bos_token=bos, add_generation_prompt=True,
    )
    limit = llm.n_ctx() - CONTEXT_RESERVE
    while True:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + history
        toks = llm.tokenize(fmt(messages=msgs).prompt.encode(), add_bos=False, special=True)
        if len(toks) <= limit or len(history) <= 1:
            return toks
        history = history[1:]  # drop oldest turn to fit context


def chat(message, cid):
    message = (message or "").strip()
    cid = (cid or "").strip()

    if not message:
        return "", get_messages(cid), cid, gr.update(
            choices=get_conversations(), value=cid or None
        )

    if not cid:
        cid = str(uuid.uuid4())[:8]

    log(f"Received prompt ({len(message)} chars) | conversation {cid}")
    log(f"VRAM before: {vram_usage()}")
    save_message(cid, "user", message)

    try:
        toks = build_tokens(get_messages(cid))
        total, n_batch = len(toks), llm.n_batch
        llm.reset()
        t0 = time.time()
        for i in range(0, total, n_batch):
            llm.eval(toks[i:i + n_batch])
            done = min(i + n_batch, total)
            log(f"Processing prompt: {done * 100 // total}% ({done}/{total} tokens)")
        log(f"Prompt processed in {time.time() - t0:.1f}s | VRAM: {vram_usage()}")

        log("Generating response...")
        t1, n, parts = time.time(), 0, []
        for chunk in llm.create_completion(
            prompt=toks, max_tokens=None, temperature=0.7,
            stop=["<|im_end|>"], stream=True,
        ):
            parts.append(chunk["choices"][0]["text"])
            n += 1
            if n % 50 == 0:
                log(f"Generating: {n} tokens ({n / (time.time() - t1):.1f} tok/s)")
        log(f"Generated {n} tokens in {time.time() - t1:.1f}s")

        answer = "".join(parts)
        answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.S)
        answer = re.sub(r"<think>.*$", "", answer, flags=re.S).strip()
        if not answer:
            answer = "(The model returned an empty response.)"

        save_message(cid, "assistant", answer)
        log("Output stored in database.")
        log(f"VRAM after: {vram_usage()}")

    except Exception as e:
        print("Model inference error:", repr(e))
        answer = f"Model error: {type(e).__name__}: {e}"
        log("Error: " + answer)

    choices = get_conversations()
    if cid not in choices:
        choices.insert(0, cid)

    return (
        "",
        get_messages(cid),
        cid,
        gr.update(choices=choices, value=cid)
    )


# ---------------- Gradio interface ----------------

with gr.Blocks(title="My AI Assistant") as demo:
    gr.Markdown("# My AI Assistant")
    gr.Markdown("Qwen3.8-27B · Chat history saved in SQLite")

    with gr.Row():
        cid = gr.Textbox(
            label="Conversation ID",
            placeholder="Create a new chat or select a saved one",
            scale=3
        )
        new_btn = gr.Button("New chat")

    saved_chats = gr.Dropdown(
        label="Saved conversations",
        choices=get_conversations(),
        interactive=True
    )

    chatbot_ui = gr.Chatbot(
    height=500,
    label="Chat"
    )

    message = gr.Textbox(
        label="Your message",
        placeholder="Message your AI...",
        lines=2
    )

    send_btn = gr.Button("Send", variant="primary")

    new_btn.click(
        new_conversation,
        outputs=[cid, chatbot_ui, saved_chats]
    )

    saved_chats.change(
        load_conversation,
        inputs=saved_chats,
        outputs=[cid, chatbot_ui]
    )

    send_btn.click(
        chat,
        inputs=[message, cid],
        outputs=[message, chatbot_ui, cid, saved_chats]
    )

    message.submit(
        chat,
        inputs=[message, cid],
        outputs=[message, chatbot_ui, cid, saved_chats]
    )

    demo.load(
        lambda: ("", []),
        outputs=[cid, chatbot_ui]
    )

print("Chat functions restored:", callable(chat))
print("Gradio interface restored:", demo is not None)


In [ ]:
import llama_cpp
from llama_cpp import Llama

if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError(
        "llama-cpp-python was installed without CUDA support. "
        "Re-run the CUDA installation cell in a GPU-enabled Colab runtime."
    )

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=32768,
    n_batch=512,
    flash_attn=True,
    verbose=False,
)
print("Qwen model loaded with GPU offload enabled.")
import subprocess
print(subprocess.check_output(["nvidia-smi","--query-gpu=index,memory.used,memory.total","--format=csv"], text=True))

In [ ]:
import os, torch

# ---- 1. GPU memory (NVML = same numbers as nvidia-smi) ----
try:
    import pynvml
    pynvml.nvmlInit()
    info = pynvml.nvmlDeviceGetMemoryInfo(pynvml.nvmlDeviceGetHandleByIndex(0))
    vram_total, vram_used, vram_free = (x / 1024**2 for x in (info.total, info.used, info.free))
except Exception:
    free_b, total_b = torch.cuda.mem_get_info(0)
    vram_total, vram_free = total_b / 1024**2, free_b / 1024**2
    vram_used = vram_total - vram_free

# ---- 2. Model file size ----
model_size_mb = os.path.getsize(MODEL_PATH) / 1024**2 if os.path.exists(MODEL_PATH) else 0

# ---- 3. Current context ----
n_ctx = llm.n_ctx() if callable(llm.n_ctx) else llm.n_ctx

# ---- 4. Real KV cache cost from GGUF metadata ----
meta = llm.metadata
arch = meta["general.architecture"]
g = lambda k, d=None: meta.get(f"{arch}.{k}", d)

n_layers = int(g("block_count"))
n_head   = int(g("attention.head_count"))
n_kv     = g("attention.head_count_kv", n_head)
emb      = int(g("embedding_length"))
key_len  = int(g("attention.key_length", emb // n_head))
val_len  = int(g("attention.value_length", key_len))

# Number of KV heads summed over layers (handles per-layer lists and hybrid models)
if isinstance(n_kv, (list, tuple)) or hasattr(n_kv, "__len__") and not isinstance(n_kv, (str, int)):
    kv_heads_total = sum(int(x) for x in n_kv)
else:
    n_kv = int(n_kv)
    interval = g("full_attention_interval")          # hybrid models only
    attn_layers = n_layers // int(interval) if interval else n_layers
    kv_heads_total = n_kv * attn_layers

elems_per_token = kv_heads_total * (key_len + val_len)
bytes_per_token = {"f16": elems_per_token * 2.0, "q8_0": elems_per_token * 34/32}

kv_now_mb = {k: v * n_ctx / 1024**2 for k, v in bytes_per_token.items()}

# ---- 5. How much context could fit? ----
# If the model is already loaded, "free" is what's left AFTER the current KV cache.
reserve_mb = 600   # compute buffers, CUDA overhead, safety margin
extra_tokens = {k: max(0, (vram_free - reserve_mb) * 1024**2 / v) for k, v in bytes_per_token.items()}
max_ctx = {k: int(n_ctx + extra_tokens[k]) for k in bytes_per_token}

print("========== VRAM ANALYSIS ==========")
print(f"Total / Used / Free VRAM  : {vram_total:,.0f} / {vram_used:,.0f} / {vram_free:,.0f} MB")
print(f"Model file size           : {model_size_mb:,.0f} MB")
print("-----------------------------------")
print(f"Arch: {arch} | layers={n_layers} | kv_heads_total={kv_heads_total} | k/v dim={key_len}/{val_len}")
print(f"KV cost per token         : f16 {bytes_per_token['f16']/1024:,.1f} KB | q8_0 {bytes_per_token['q8_0']/1024:,.1f} KB")
print(f"KV cache at n_ctx={n_ctx:,}  : f16 {kv_now_mb['f16']:,.0f} MB | q8_0 {kv_now_mb['q8_0']:,.0f} MB")
print("-----------------------------------")
print(f"Max context that should fit (keeping {reserve_mb} MB spare):")
print(f"  with f16 KV cache       : ~{max_ctx['f16']:,} tokens")
print(f"  with q8_0 KV cache      : ~{max_ctx['q8_0']:,} tokens")
print("===================================")

In [12]:

import os
import time
import socket
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

# 1. Read Kaggle Secrets (Add-ons -> Secrets)
_s = UserSecretsClient()
ngrok_token = _s.get_secret("NGROK_AUTHTOKEN")
ngrok_domain = _s.get_secret("NGROK_DOMAIN")

if not ngrok_token or not ngrok_domain:
    raise RuntimeError(
        "Add NGROK_AUTHTOKEN and NGROK_DOMAIN to Kaggle Secrets (Add-ons -> Secrets)."
    )

# Normalize domain
ngrok_domain = ngrok_domain.strip().rstrip("/")
ngrok_domain = (
    ngrok_domain
    .removeprefix("https://")
    .removeprefix("http://")
    .split("/")[0]
)

# 2. Configure authentication
try:
    ngrok.set_auth_token(ngrok_token)
except Exception as e:
    if "Authtoken already set in configuration file" not in str(e):
        raise

print("Ngrok authentication ready.")

# 3. Check that your existing chatbot interface exists
if "demo" not in globals():
    raise RuntimeError(
        "The variable 'demo' does not exist. Run the cell that "
        "defines your Gradio chatbot interface first."
    )

if "llm" not in globals():
    raise RuntimeError(
        "The variable 'llm' does not exist. Run your model-loading "
        "cell first; do not reinstall packages unnecessarily."
    )

# 4. Start Gradio on a free port
sock = socket.socket()
sock.bind(("127.0.0.1", 0))
port = sock.getsockname()[1]
sock.close()

print("Starting Gradio on port:", port)

demo.launch(
    server_name="0.0.0.0",
    server_port=port,
    share=False,
    prevent_thread_lock=True,
    theme=gr.themes.Soft()
)

# 5. Wait for Gradio to become available
for _ in range(30):
    try:
        with socket.create_connection(
            ("127.0.0.1", port), timeout=2
        ):
            print("Gradio is ready.")
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError(
        "Gradio did not start. Check the preceding launch output."
    )

# 6. Connect ngrok to the reserved domain
# Stop here with a clear error if the endpoint API rejects the config.
try:
    tunnel = ngrok.connect(
        addr=port,
        proto="http",
        url="https://" + ngrok_domain
    )

    print("\n" + "=" * 50)
    print("CHATBOT ONLINE")
    print("Public URL:", tunnel.public_url)
    print("=" * 50)

except Exception as e:
    print("Ngrok tunnel failed:", repr(e))
    print(
        "Gradio is running locally, but the public tunnel "
        "still needs to be configured."
    )
    raise


In [ ]:
print("Model loaded:", "llm" in globals())
print("Chat functions loaded:", "chat" in globals())
print("Database path defined:", "CHAT_DB" in globals())
print("Gradio interface defined:", "demo" in globals())